# RewindKV — Experimento Go/No-Go (Fase 1)

Compara **Baseline**, **Blind Rewind** e **RewindKV** com o `DeepSeek-R1-Distill-Qwen-1.5B`.

**Antes de rodar:** `Ambiente de execução > Alterar o tipo de ambiente > GPU T4`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código do projeto
O repositório é **privado**, então o Colab precisa de um token (só uma vez):
1. Em <https://github.com/settings/personal-access-tokens/new> crie um *fine-grained token* com acesso só ao repositório `REWIND-KV` e permissão **Contents: Read-only**.
2. No Colab, clique no ícone 🔑 (*Secrets*) na barra lateral, adicione `GITHUB_TOKEN` com o token e ative o acesso para este notebook.

Rodar a célula de novo faz `git pull`, puxando as últimas mudanças. Se deixar `REPO` vazio, ela pede o upload de um `.zip` da pasta.

In [ ]:
import os, zipfile, glob

REPO = "GioNicolas/REWIND-KV"

%cd /content
if REPO and os.path.isdir("REWIND-KV/.git"):
    !git -C REWIND-KV pull -q
elif REPO:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    !git clone -q https://{token}@github.com/{REPO}.git REWIND-KV
else:
    from google.colab import files
    uploaded = files.upload()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall("/content")

project = os.path.dirname(glob.glob("/content/**/pyproject.toml", recursive=True)[0])
%cd {project}
!pip install -q -e ".[dev]"

## 2. Sanidade: testes unitários do crop/RoPE e do detector

In [ ]:
!python -m pytest -q

## 3. Rodada piloto (rápida)
5 problemas de nível 4, só **baseline** e **rewind_kv** (~30 min). Serve para conferir com que frequência o modelo faz backtracking e se os cortes acertam o passo errado, antes de gastar horas de GPU.

*Configuração da v0.1 (corte por parágrafo, âncora `[Note: ...]`), para reproduzir os pilotos.*

In [ ]:
!python -m rewind_kv.experiment --dataset math500 --level 4 --n 5 --max-new-tokens 4096 --modes baseline rewind_kv --rewind-scope paragraph --anchor-style note

## 4. Experimento A (acurácia): 30 problemas
Baseline vs RewindKV em 30 problemas de nível 4 (~2–3 h na T4).

Os resultados vão direto para o seu **Google Drive** (`MyDrive/rewindkv_results`), linha a linha. Se o Colab desconectar, nada se perde: veja quantos problemas já terminaram (`[N/30]` na última linha) e rode a célula de novo trocando `START = 0` por `START = N`.

*Já rodado com a v0.1 (18/10/2026). Para a v0.2, vá para a seção 6.*

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/rewindkv_results"

START = 0  # se cair, troque pelo nº de problemas que já terminaram
!python -m rewind_kv.experiment --dataset math500 --level 4 --start {START} --n {30 - START} --max-new-tokens 4096 --modes baseline rewind_kv --rewind-scope paragraph --anchor-style note --out {OUT}

## 5. Análise
O critério Go/No-Go só vale para os problemas em que o **baseline fez pelo menos 1 backtracking** (`detections >= 1`).

In [ ]:
import json, glob
import pandas as pd

# Junta todas as partes do experimento no Drive (inclusive retomadas); sem Drive, usa o último arquivo local.
paths = sorted(glob.glob("/content/drive/MyDrive/rewindkv_results/*.jsonl")) or sorted(glob.glob("results/*.jsonl"))[-1:]
rows = [json.loads(l) for p in paths for l in open(p, encoding="utf-8")]
df = pd.DataFrame(rows).drop_duplicates(["problem_idx", "mode"], keep="last")
print(f"{len(paths)} arquivo(s), {df['problem_idx'].nunique()} problemas")
df["n_rewinds"] = df["rewinds"].apply(len)
df["discarded_tokens"] = df["rewinds"].apply(lambda rs: sum(r["discarded_tokens"] for r in rs))

backtracked = df[(df["mode"] == "baseline") & (df["detections"] >= 1)]["problem_idx"]
sub = df[df["problem_idx"].isin(backtracked)]
print(f"{len(backtracked)} de {df['problem_idx'].nunique()} problemas tiveram backtracking no baseline\n")

cols = ["correct", "final_cache_len", "peak_cache_len", "generated_tokens", "n_rewinds",
        "discarded_tokens", "repeated_failures", "hit_token_limit", "final_cache_mb", "seconds"]
print("Todos os problemas:")
display(df.groupby("mode")[cols].mean())
print("\nSó os problemas com backtracking no baseline:")
table = sub.groupby("mode")[cols].mean()
display(table)

if {"baseline", "rewind_kv"} <= set(table.index):
    red = 1 - table.loc["rewind_kv", "final_cache_len"] / table.loc["baseline", "final_cache_len"]
    print(f"Redução do cache final (rewind_kv vs baseline): {red:.1%}")
    print(f"Acurácia: baseline={table.loc['baseline', 'correct']:.1%}  rewind_kv={table.loc['rewind_kv', 'correct']:.1%}")

### Inspecionar detecções e âncoras
Leia à mão: cada detecção do baseline deveria ser um erro de verdade, e cada âncora deveria nomear o passo errado.

In [ ]:
for _, row in df[df["mode"] == "baseline"].iterrows():
    for reason in row["detected_reasons"]:
        print(f"[problema {row['problem_idx']}] detectou: {reason}")
print()
for _, row in df[df["mode"] == "rewind_kv"].iterrows():
    for ev in row["rewinds"]:
        print(f"[problema {row['problem_idx']}] cortou {ev['discarded_tokens']} tokens -> âncora de {ev['anchor_tokens']}:")
        print("   ", ev["anchor"].strip(), "\n")

## 6. Experimento A2 (v0.2): os 10 problemas com backtracking
No Experimento A, o modelo **refazia a conta cortada** logo depois da âncora: só o parágrafo do "errei" era cortado, e as premissas do caminho errado continuavam no cache. A v0.2 muda duas coisas:
- **Corte por caminho** (`--rewind-scope branch`): volta até a frase que abriu o caminho ("Alternatively, ...", "Let me try ...").
- **Âncora na voz do modelo** (`--anchor-style voice`): *Wait, I already tried this before: "..." It was wrong: ... I will not repeat it, so let me do it differently.*

Roda só **rewind_kv** e **blind_rewind** (o baseline é o do Experimento A, mesma semente) nos 10 problemas em que o baseline fez backtracking: ~50 min na T4. Esses 10 problemas são o **conjunto de desenvolvimento**; se a v0.2 funcionar, a validação é em problemas novos.

Se o Colab cair, tire da lista `PROBLEMS` os que já terminaram (aparecem como `p0`, `p3`... na saída) e rode de novo.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT2 = "/content/drive/MyDrive/rewindkv_results_v02"

PROBLEMS = "0 3 5 6 13 15 17 22 24 26"
!python -m rewind_kv.experiment --dataset math500 --level 4 --problems {PROBLEMS} --max-new-tokens 4096 --modes rewind_kv blind_rewind --rewind-scope branch --anchor-style voice --out {OUT2}

### Análise do A2
Compara, problema a problema, o baseline e o rewind_kv v0.1 (Experimento A) com o rewind_kv e o blind_rewind da v0.2.

A coluna `prefixo_ok` confere se a geração foi idêntica ao baseline até o primeiro corte (mesma semente). Se der `False`, a GPU gerou diferente nesta sessão e a comparação com o baseline antigo deixa de ser pareada.

In [ ]:
import json, glob
import pandas as pd

def load(pattern):
    rows = [json.loads(l) for p in sorted(glob.glob(pattern)) for l in open(p, encoding="utf-8")]
    return pd.DataFrame(rows)

a = load("/content/drive/MyDrive/rewindkv_results/*.jsonl").drop_duplicates(["problem_idx", "mode"], keep="last")
v2 = load("/content/drive/MyDrive/rewindkv_results_v02/*.jsonl").drop_duplicates(["problem_idx", "mode"], keep="last")
base = a[a["mode"] == "baseline"].set_index("problem_idx")
v1 = a[a["mode"] == "rewind_kv"].set_index("problem_idx")

def prefix_ok(row):
    if not row["rewinds"]:
        return row["text"] == base.loc[row["problem_idx"], "text"]
    b = base.loc[row["problem_idx"], "text"]
    c = b.find(row["rewinds"][0]["discarded_text"][:80])
    return c >= 0 and row["text"][:c] == b[:c]

v2["prefixo_ok"] = v2.apply(prefix_ok, axis=1)
v2["n_rewinds"] = v2["rewinds"].apply(len)

table = pd.DataFrame({"baseline": base["correct"], "v0.1": v1["correct"]}).loc[sorted(v2["problem_idx"].unique())]
for mode in ["rewind_kv", "blind_rewind"]:
    m = v2[v2["mode"] == mode].set_index("problem_idx")
    table[f"v0.2 {mode}"] = m["correct"]
    table[f"{mode} tokens"] = m["generated_tokens"]
    table[f"{mode} rewinds"] = m["n_rewinds"]
    table[f"{mode} prefixo_ok"] = m["prefixo_ok"]
table["baseline tokens"] = base["generated_tokens"]
display(table)
print("Acertos:", {c: int(table[c].sum()) for c in ["baseline", "v0.1", "v0.2 rewind_kv", "v0.2 blind_rewind"] if c in table})

### O modelo refez a conta cortada?
Para cada corte: o começo do trecho cortado, a âncora e o que o modelo escreveu logo depois. Se o "depois" repete o "cortado", a âncora não funcionou.

In [ ]:
for _, row in v2.sort_values(["problem_idx", "mode"]).iterrows():
    for ev in row["rewinds"]:
        after = row["text"][ev["resume_char"] : ev["resume_char"] + 300]
        print(f"[p{row['problem_idx']} {row['mode']}] cortou {ev['discarded_tokens']} tokens, âncora de {ev['anchor_tokens']}")
        print("  cortado:", ev["discarded_text"][:200].replace("\n", " "))
        if ev["anchor"]:
            print("  âncora: ", ev["anchor"].strip())
        print("  depois: ", after.replace("\n", " "), "\n")

In [ ]:
# Só é preciso se você rodou sem o Drive: baixa os resultados locais.
from google.colab import files
for p in glob.glob("results/*.jsonl"):
    files.download(p)